# Mirror-Prompted Dataset Builder — Free API Version

This notebook generates AI mirror essays using **free API tiers** — no payment required.

## Free provider options

| Provider | Model | Free limit | Sign-up |
|----------|-------|-----------|--------|
| **Google Gemini** | gemini-1.5-flash | 15 req/min, 1M tokens/day | [aistudio.google.com](https://aistudio.google.com) |
| **Groq** | llama-3.1-70b-versatile | ~30 req/min | [console.groq.com](https://console.groq.com) |
| **Ollama** | llama3 / mistral (local) | Unlimited | [ollama.com](https://ollama.com) |

**Recommendation:** Run with Gemini first, then re-run with Groq to get two sets of mirrors from different model families. You'll end up with multi-model diversity for free — exactly what Pangram did with GPT, Gemini, Mistral, and LLaMA.

## Time estimate for 84 essays
- **Gemini free tier** (15 req/min): ~12 minutes
- **Groq** (~30 req/min): ~6 minutes  
- **Ollama local** (depends on hardware): 30–90 minutes


## 0) Install dependencies

In [16]:
# Uncomment the provider(s) you want to use:
!pip install -q google-generativeai pandas tqdm  # Gemini
!pip install -q groq pandas tqdm               # Groq
# !pip install -q ollama pandas tqdm             # Ollama (also needs Ollama server running)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.1 MB/s eta 0:00:00


## 1) Configuration — pick your provider here

In [17]:
import os, re, json, time, random, unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# ============================================================
# PICK YOUR PROVIDER
# ============================================================
PROVIDER = "groq"   # Options: "gemini", "groq", "ollama"

# ---- Gemini settings ----
GEMINI_MODEL = "gemini-2.0-flash"   # free tier model
# Get your free key at: https://aistudio.google.com/app/apikey
# In Colab: add GEMINI_API_KEY to Secrets (left sidebar key icon)

# ---- Groq settings ----
GROQ_MODEL = "llama-3.3-70b-versatile"  # or "mixtral-8x7b-32768"
# Get your free key at: https://console.groq.com
# In Colab: add GROQ_API_KEY to Secrets

# ---- Ollama settings (fully local, no API key needed) ----
OLLAMA_MODEL   = "llama3"          # or "mistral", "llama3:70b" if you have VRAM
OLLAMA_HOST    = "http://localhost:11434"  # default Ollama server address

# ============================================================
# DATA & OUTPUT
# ============================================================
DATA_PATH   = "essays.csv"
TEXT_COL    = "essay"
OUTPUT_DIR  = "."          # outputs saved here

MIN_CHAR_LEN = 500
MAX_CHAR_LEN = 12_000
MAX_SAMPLES  = None        # None = all essays; set e.g. 10 for a quick test
SEED         = 42

# ============================================================
# RATE LIMITING  (adjusted per provider)
# ============================================================
RATE_LIMITS = {
    # Gemini free: 15 req/min → one call every 4.5s to stay safe
    "gemini": {"inter_call_sleep": 4.5, "max_retries": 5, "retry_base_delay": 10.0},
    # Groq free: generous limits
    "groq":   {"inter_call_sleep": 2.0, "max_retries": 4, "retry_base_delay": 5.0},
    # Ollama local: no rate limit
    "ollama": {"inter_call_sleep": 0.1, "max_retries": 3, "retry_base_delay": 2.0},
}

rl = RATE_LIMITS[PROVIDER]
INTER_CALL_SLEEP = rl["inter_call_sleep"]
MAX_RETRIES      = rl["max_retries"]
RETRY_BASE_DELAY = rl["retry_base_delay"]

MAX_TOKENS_TITLE = 60
MAX_TOKENS_ESSAY = 1500
TEMPERATURE      = 0.7

RESUME_FROM_LOG  = None   # set to log path to resume after a crash

random.seed(SEED)
print(f"Provider    : {PROVIDER}")
print(f"Data path   : {DATA_PATH}")
print(f"Max samples : {MAX_SAMPLES if MAX_SAMPLES else 'all'}")

Provider    : groq
Data path   : essays.csv
Max samples : all


In [18]:
# Model Verifier
import google.generativeai as genai

# List available models
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025

## 2) Load API key and initialise client

In [19]:
def _load_secret(name: str) -> str:
    """Try Colab Secrets first, then environment variables."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            print(f"Loaded {name} from Colab Secrets")
            return val
    except Exception:
        pass
    val = os.environ.get(name, "")
    if val:
        print(f"Loaded {name} from environment variable")
    return val


# ---- Initialise the chosen provider ----

if PROVIDER == "gemini":
    import google.generativeai as genai

    api_key = _load_secret("GEMINI_API_KEY")
    assert api_key, (
        "GEMINI_API_KEY not found.\n"
        "Get a free key at https://aistudio.google.com/app/apikey\n"
        "Then add it to Colab Secrets or set as environment variable."
    )
    genai.configure(api_key=api_key)
    _gemini_model = genai.GenerativeModel(GEMINI_MODEL)
    MODEL_LABEL = GEMINI_MODEL
    print(f"Gemini client ready: {GEMINI_MODEL}")


elif PROVIDER == "groq":
    from groq import Groq

    api_key = _load_secret("GROQ_API_KEY")
    assert api_key, (
        "GROQ_API_KEY not found.\n"
        "Get a free key at https://console.groq.com\n"
        "Then add it to Colab Secrets or set as environment variable."
    )
    _groq_client = Groq(api_key=api_key)
    MODEL_LABEL  = GROQ_MODEL
    print(f"Groq client ready: {GROQ_MODEL}")


elif PROVIDER == "ollama":
    import urllib.request

    # Verify Ollama server is reachable
    try:
        urllib.request.urlopen(OLLAMA_HOST, timeout=3)
        print(f"Ollama server reachable at {OLLAMA_HOST}")
    except Exception:
        raise RuntimeError(
            f"Cannot reach Ollama server at {OLLAMA_HOST}.\n"
            "Make sure Ollama is installed and running: https://ollama.com\n"
            "Then pull a model: ollama pull llama3"
        )
    MODEL_LABEL = OLLAMA_MODEL
    print(f"Ollama ready: {OLLAMA_MODEL}")


else:
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}. Choose: gemini, groq, ollama")

Loaded GROQ_API_KEY from Colab Secrets
Groq client ready: llama-3.3-70b-versatile


## 3) Unified `call_llm()` function

Single function that routes to the correct provider. All retry / rate-limit logic lives here.

In [20]:
def call_llm(prompt: str, max_tokens: int, temperature: float = 0.7) -> str | None:
    for attempt in range(MAX_RETRIES):
        try:
            if PROVIDER == "gemini":
                cfg = genai.types.GenerationConfig(
                    max_output_tokens=max_tokens,
                    temperature=temperature,
                )
                resp = _gemini_model.generate_content(prompt, generation_config=cfg)
                if not resp.candidates:
                    print("  Gemini safety filter triggered. Skipping.")
                    return None
                return resp.text

            elif PROVIDER == "groq":
                resp = _groq_client.chat.completions.create(
                    model=GROQ_MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tokens,
                    temperature=temperature,
                )
                return resp.choices[0].message.content

            elif PROVIDER == "ollama":
                import urllib.request as ur
                payload = json.dumps({
                    "model": OLLAMA_MODEL, "prompt": prompt,
                    "stream": False,
                    "options": {"num_predict": max_tokens, "temperature": temperature},
                }).encode()
                req = ur.Request(
                    f"{OLLAMA_HOST}/api/generate", data=payload,
                    headers={"Content-Type": "application/json"}, method="POST",
                )
                with ur.urlopen(req, timeout=180) as r:
                    return json.loads(r.read()).get("response", "")

        except Exception as e:
            # Get the full exception info including class name
            exc_type = type(e).__name__.lower()
            msg = str(e).lower()
            full = f"{exc_type} {msg}"

            # Never retry these — they won't resolve with waiting
            if any(kw in full for kw in ["404", "not found", "invalid", "authentication", "permission"]):
                print(f"  Fatal error (won't retry): {type(e).__name__}: {e}")
                return None

            # Retry these — temporary server/rate-limit issues
            is_retryable = any(kw in full for kw in [
                "429", "toomany", "rate", "quota", "resource exhausted",
                "500", "502", "503", "unavailable", "timeout", "connection"
            ])

            if is_retryable:
                wait = RETRY_BASE_DELAY * (2 ** attempt) + random.uniform(0, 2)
                print(f"  Retryable error [{type(e).__name__}] "
                      f"(attempt {attempt+1}/{MAX_RETRIES}). Waiting {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"  Unknown error (won't retry): {type(e).__name__}: {e}")
                return None

    print(f"  All {MAX_RETRIES} retries exhausted.")
    return None


# Connectivity test — skipped if quota is already exhausted
print("Running connectivity test...")
test = call_llm("Reply with exactly the word OK and nothing else.", max_tokens=5, temperature=0.0)
if test:
    print(f"Connectivity test passed. Response: {repr(test.strip())}")
else:
    print("Connectivity test failed — but continuing anyway. The main loop may still work.")
    print("If using Gemini, your per-minute quota may just be temporarily exhausted.")
    print("Wait 60 seconds and re-run the main generation loop cell directly.")

Running connectivity test...
Connectivity test passed. Response: 'OK'


## 4) Load human essays

In [21]:
assert os.path.exists(DATA_PATH), f"Cannot find {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)
print(f"Raw shape : {df_raw.shape}  |  Columns: {list(df_raw.columns)}")

assert TEXT_COL in df_raw.columns

df_human = (
    df_raw[[TEXT_COL]]
    .copy()
    .rename(columns={TEXT_COL: "text"})
    .assign(text=lambda d: d["text"].astype(str).str.strip())
    .pipe(lambda d: d[d["text"].str.len().between(MIN_CHAR_LEN, MAX_CHAR_LEN)])
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

if MAX_SAMPLES and len(df_human) > MAX_SAMPLES:
    df_human = df_human.sample(MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

df_human["word_count"] = df_human["text"].apply(lambda t: len(t.split()))

print(f"After filtering: {len(df_human)} essays")
print(df_human["word_count"].describe().to_string())

Raw shape : (2235, 6)  |  Columns: ['title', 'description', 'essay', 'authors', 'source_url', 'thumbnail_url']
After filtering: 84 essays
count      84.00000
mean     1715.50000
std       249.88877
min       546.00000
25%      1577.25000
50%      1759.50000
75%      1899.25000
max      2174.00000


## 5) Cleaning utilities & prompt templates

In [22]:
# ---- Cleaning ----

_BOILERPLATE_PREFIXES = [
    r"^Sure[,!.]?\s*",
    r"^Here(?:'s| is) (?:a|an|the|my) .{0,60}?:\s*",
    r"^Title:\s*.+?\n",
    r"^(?:Of course|Certainly|Absolutely)[,!.]?\s*(?:Here .{0,60}?:\s*)?",
    r"^I(?:'m| am) happy to help.{0,80}?:\s*",
    r"^As requested,.{0,80}?:\s*",
    r"^Below (?:is|are) .{0,60}?:\s*",
]
_BOILERPLATE_SUFFIXES = [
    r"\nI hope (?:this|that) .{0,120}?$",
    r"\nLet me know if .{0,120}?$",
    r"\nFeel free to .{0,120}?$",
    r"\n\(Word count: \d+\s*words?\)\s*$",
    r"\n---+\s*$",
]
_EMOJI_RE = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+",
    flags=re.UNICODE,
)
_QUOTE_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201C": '"', "\u201D": '"',
    "\u2013": "-", "\u2014": "-", "\u2026": "...",
})

def clean_llm_output(text: str, min_words: int = 50) -> str | None:
    if not text:
        return None
    text = unicodedata.normalize("NFKC", text)
    text = _EMOJI_RE.sub("", text)
    text = text.translate(_QUOTE_MAP)
    for pat in _BOILERPLATE_PREFIXES:
        text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
    for pat in _BOILERPLATE_SUFFIXES:
        text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text if len(text.split()) >= min_words else None

def clean_title(raw: str) -> str:
    t = raw.strip()
    t = re.sub(r'^["\']|["\']$', "", t).strip()
    t = re.sub(r'^(?:Title|Subject):\s*', "", t, flags=re.IGNORECASE).strip()
    return t


# ---- Prompts ----

def make_title_prompt(essay_text: str) -> str:
    excerpt = " ".join(essay_text.split()[:800])
    return (
        "What is a concise, descriptive title for the following essay?\n\n"
        f"{excerpt}\n\n"
        "Respond with ONLY the title — no quotation marks, no explanation, no preamble."
    )

def make_mirror_prompt(title: str, target_word_count: int) -> str:
    low  = int(target_word_count * 0.85)
    high = int(target_word_count * 1.15)
    return (
        f"Write an essay with the following title:\n\nTitle: {title}\n\n"
        f"Requirements:\n"
        f"- Length: between {low} and {high} words\n"
        f"- Write in a thoughtful, personal voice as if written by a knowledgeable human author\n"
        f"- Do NOT include the title at the top of your response\n"
        f"- Do NOT include any preamble, word count, headers, or closing remarks\n"
        f"- Begin directly with the first sentence of the essay body"
    )

print("Cleaning utilities and prompt templates ready.")

Cleaning utilities and prompt templates ready.


## 6) Main generation loop

In [ ]:
def load_completed(log_path) -> set:
    done = set()
    if not log_path or not os.path.exists(log_path):
        return done
    with open(log_path) as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get("status") == "success":
                    done.add(r["essay_idx"])
            except json.JSONDecodeError:
                pass
    print(f"Resuming: {len(done)} essays already done.")
    return done


# Output paths — include provider name so multi-provider runs don't collide
log_path = os.path.join(OUTPUT_DIR, f"mirror_log_{PROVIDER}.jsonl")
already_done = load_completed(RESUME_FROM_LOG)

results = []
# Pre-fill results from existing log if resuming
if RESUME_FROM_LOG and os.path.exists(RESUME_FROM_LOG):
    with open(RESUME_FROM_LOG) as f:
        for line in f:
            try:
                r = json.loads(line)
                if r.get("status") == "success":
                    results.append(r)
            except json.JSONDecodeError:
                pass

stats = {"attempted": 0, "success": 0,
         "title_fail": 0, "mirror_fail": 0, "clean_fail": 0}

log_handle = open(log_path, "a")
start_time = time.time()

for idx, row in tqdm(df_human.iterrows(), total=len(df_human), desc=f"Generating mirrors [{PROVIDER}]"):

    if idx in already_done:
        continue

    stats["attempted"] += 1
    human_text = row["text"]
    word_count = row["word_count"]
    rec = {"essay_idx": idx, "timestamp": datetime.utcnow().isoformat(),
           "provider": PROVIDER, "model": MODEL_LABEL, "word_count": word_count}

    # Step 1: Title
    raw_title = call_llm(make_title_prompt(human_text), max_tokens=MAX_TOKENS_TITLE, temperature=0.3)
    time.sleep(INTER_CALL_SLEEP)

    if not raw_title:
        stats["title_fail"] += 1
        rec["status"] = "title_fail"
        log_handle.write(json.dumps(rec) + "\n"); log_handle.flush()
        continue

    title = clean_title(raw_title)
    rec["title"] = title

    # Step 2: Mirror essay
    raw_mirror = call_llm(make_mirror_prompt(title, word_count), max_tokens=MAX_TOKENS_ESSAY, temperature=TEMPERATURE)
    time.sleep(INTER_CALL_SLEEP)

    if not raw_mirror:
        stats["mirror_fail"] += 1
        rec["status"] = "mirror_fail"
        log_handle.write(json.dumps(rec) + "\n"); log_handle.flush()
        continue

    # Step 3: Clean
    cleaned = clean_llm_output(raw_mirror, min_words=50)

    if not cleaned:
        stats["clean_fail"] += 1
        rec["status"] = "clean_fail"
        rec["raw_mirror_snippet"] = raw_mirror[:200]
        log_handle.write(json.dumps(rec) + "\n"); log_handle.flush()
        continue

    # Success
    stats["success"] += 1
    rec.update({
        "status":        "success",
        "human_text":    human_text,
        "ai_text":       cleaned,
        "ai_word_count": len(cleaned.split()),
        "raw_title":     raw_title,
    })
    log_handle.write(json.dumps(rec) + "\n"); log_handle.flush()
    results.append(rec)

log_handle.close()
elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} min. Stats: {stats}")

Generating mirrors [groq]:   0%|          | 0/84 [00:00<?, ?it/s]

/tmp/ipykernel_2608/2901396859.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rec = {"essay_idx": idx, "timestamp": datetime.utcnow().isoformat(),


  Retryable error [RateLimitError] (attempt 1/4). Waiting 6.3s...
  Retryable error [RateLimitError] (attempt 2/4). Waiting 10.1s...
  Retryable error [RateLimitError] (attempt 3/4). Waiting 20.6s...
  Retryable error [RateLimitError] (attempt 4/4). Waiting 40.4s...
  All 4 retries exhausted.
  Retryable error [RateLimitError] (attempt 1/4). Waiting 6.5s...
  Retryable error [RateLimitError] (attempt 2/4). Waiting 11.4s...
  Retryable error [RateLimitError] (attempt 3/4). Waiting 21.8s...
  Retryable error [RateLimitError] (attempt 4/4). Waiting 40.2s...
  All 4 retries exhausted.
  Retryable error [RateLimitError] (attempt 1/4). Waiting 5.8s...
  Retryable error [RateLimitError] (attempt 2/4). Waiting 10.1s...
  Retryable error [RateLimitError] (attempt 3/4). Waiting 20.4s...
  Retryable error [RateLimitError] (attempt 4/4). Waiting 41.0s...
  All 4 retries exhausted.
  Retryable error [RateLimitError] (attempt 1/4). Waiting 5.1s...
  Retryable error [RateLimitError] (attempt 2/4). Wa

## 7) Build the final CSV

In [ ]:
rows = []
for rec in results:
    if rec.get("status") != "success":
        continue
    rows.append({"text": rec["human_text"], "label": 1, "source": "human",
                 "title": rec["title"], "word_count": rec["word_count"], "pair_id": rec["essay_idx"]})
    rows.append({"text": rec["ai_text"], "label": 0, "source": f"ai_{PROVIDER}_{MODEL_LABEL}",
                 "title": rec["title"], "word_count": rec["ai_word_count"], "pair_id": rec["essay_idx"]})

df_out = pd.DataFrame(rows).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

out_csv = os.path.join(OUTPUT_DIR, f"mirror_dataset_{PROVIDER}.csv")
df_out.to_csv(out_csv, index=False)

print(f"Saved {len(df_out)} rows to {out_csv}")
print(df_out["label"].value_counts().to_string())
df_out.head(4)

## 8) (Optional) Merge multi-provider datasets

If you've run this notebook with both Gemini and Groq, merge the CSVs here.
Each human essay appears once; each model's mirror appears once — giving you
multi-LLM diversity exactly like Pangram.

In [ ]:
provider_csvs = [
    "mirror_dataset_gemini.csv",
    "mirror_dataset_groq.csv",
    # "mirror_dataset_ollama.csv",   # uncomment if you ran Ollama too
]

available = [p for p in provider_csvs if os.path.exists(p)]

if len(available) < 2:
    print(f"Only {len(available)} provider CSV(s) found — nothing to merge yet.")
    print(f"Available: {available}")
else:
    dfs = [pd.read_csv(p) for p in available]

    # Keep only one copy of each human essay (they're identical across CSVs)
    human_df = dfs[0][dfs[0].label == 1].drop_duplicates(subset=["pair_id"])

    # Collect all AI mirrors from all providers
    ai_dfs = [d[d.label == 0] for d in dfs]

    merged = pd.concat([human_df] + ai_dfs, ignore_index=True)
    merged = merged.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    merged_path = "mirror_dataset_merged.csv"
    merged.to_csv(merged_path, index=False)

    print(f"Merged dataset: {merged.shape}")
    print(merged["source"].value_counts().to_string())
    print(f"\nSaved to {merged_path}")
    print("\nUse 'mirror_dataset_merged.csv' in your classifier notebook for maximum diversity.")

## 9) Inspect a sample pair

In [ ]:
from IPython.display import HTML, display

def show_pair(df, pair_id: int, char_limit: int = 700):
    pair  = df[df.pair_id == pair_id]
    human = pair[pair.label == 1].iloc[0]
    ai    = pair[pair.label == 0].iloc[0]
    display(HTML(f"""
    <div style="display:flex;gap:20px;font-family:Georgia,serif;font-size:14px;">
      <div style="flex:1;border:2px solid steelblue;padding:12px;border-radius:6px;">
        <b style="color:steelblue;">HUMAN</b>
        <div style="color:#888;font-size:12px;">Title: {human['title']} | {human['word_count']} words</div><hr/>
        {human['text'][:char_limit]}{'...' if len(human['text']) > char_limit else ''}
      </div>
      <div style="flex:1;border:2px solid tomato;padding:12px;border-radius:6px;">
        <b style="color:tomato;">AI MIRROR ({ai['source']})</b>
        <div style="color:#888;font-size:12px;">Title: {ai['title']} | {ai['word_count']} words</div><hr/>
        {ai['text'][:char_limit]}{'...' if len(ai['text']) > char_limit else ''}
      </div>
    </div>"""))

sample_ids = df_out["pair_id"].unique()[:3]
for pid in sample_ids:
    print(f"\n{'='*80}")
    show_pair(df_out, pid)

## 10) Next steps

### Using this dataset in your classifier notebook

Replace the `corrupt_to_slop` section in your classifier notebook with:

```python
import pandas as pd
from sklearn.model_selection import train_test_split

# Use single-provider or merged dataset:
df_all = pd.read_csv("mirror_dataset_gemini.csv")     # single provider
# df_all = pd.read_csv("mirror_dataset_merged.csv")   # multi-provider (better)

train_df, temp_df = train_test_split(df_all, test_size=0.30, random_state=42, stratify=df_all["label"])
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])
# Everything else in the classifier notebook is unchanged.
```

### Getting more data for free
Run this notebook a second time with `PROVIDER = "groq"` to get Llama 3 mirrors.  
Then merge both CSVs in Section 8 above.

### Getting even more human essays (free)
- **PERSUADE corpus** (student writing): https://github.com/scrosseye/persuade_corpus_2.0  
- **Reddit WritingPrompts** (HuggingFace): `datasets.load_dataset('euclaise/writingprompts')`  
- **Project Gutenberg** (books): https://www.gutenberg.org

Add them to your `essays.csv` and re-run this notebook — the resumption logic skips already-processed essays.
